# CIFAR-10: Transfer Learning con Modelos Preentrenados

En la clase 05 entrenamos una DenseNet **desde cero** sobre CIFAR-10 y alcanzamos aproximadamente un 84% de accuracy después de 30 épocas. Todo lo que la red sabe lo aprendió de las 40,000 imágenes de entrenamiento. En esta notebook partimos de otra idea: reutilizar una red que ya fue entrenada sobre un conjunto de datos mucho más grande (**ImageNet**, 1.28 millones de imágenes) y adaptarla a nuestro problema. Esta técnica se llama **transfer learning** y es, en la práctica, la forma más común de entrenar un modelo de visión cuando los datos propios son pocos.

## Introducción

### Objetivos

1. **Entender qué se transfiere** de una red preentrenada y por qué es útil en otro conjunto de datos.
2. **Utilizar modelos preentrenados de `torchvision`** y adaptarlos a CIFAR-10: la cabeza de clasificación, la normalización y la resolución de entrada.
3. **Comparar tres estrategias con la misma arquitectura** (ResNet-18): entrenamiento desde cero, *fine-tuning parcial* y *fine-tuning*, en accuracy, parámetros entrenables y tiempo.
4. **Analizar cuándo conviene** el transfer learning, en particular cuando el conjunto de entrenamiento es pequeño.

### Contenido

1. Configuración de bibliotecas y semillas para reproducibilidad.
2. Qué es transfer learning.
3. El modelo preentrenado: arquitectura de ResNet-18, sus predicciones sobre CIFAR-10, sus filtros y sus mapas de características.
4. Preprocesamiento: normalización y resolución de entrada.
5. Tres estrategias: desde cero, fine-tuning parcial y fine-tuning.
6. Comparación en validación y experimento con pocos datos.
7. Evaluación final en test, conclusiones y ejercicios.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import v2 as T
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.utils import make_grid

from torchinfo import summary

import time
from pathlib import Path
import matplotlib.pyplot as plt

from utils import (
    train,
    print_log,
    accuracy,
    model_classification_report,
    show_tensor_images,
)

In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
# cuDNN elige algoritmos deterministas (mismo resultado en cada corrida, algo más lento)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from utils import get_device, get_num_workers

DEVICE = get_device()  # cuda > mps > xpu > cpu
# Linux: mitad de los núcleos disponibles, entre 1 y 8; Windows/macOS: 0 (ver docstring)
NUM_WORKERS = get_num_workers()

print(f"Usando {DEVICE}")
print(f"Usando {NUM_WORKERS} workers")

In [ ]:
# menor que en la clase 05 (512): las imágenes se amplían a 128x128 (ver más abajo),
# cada una ocupa 16 veces más memoria que a 32x32
BATCH_SIZE = 256

## ¿Qué es transfer learning?

Una CNN entrenada con fotografías aprende una **jerarquía de características**. Las primeras capas detectan bordes orientados, cambios de color y texturas simples; las capas intermedias combinan esos patrones en partes (ojos, ruedas, hojas); las últimas capas responden a objetos completos de las clases del entrenamiento. [Zeiler y Fergus (2014)](https://arxiv.org/abs/1311.2901) visualizaron esta jerarquía y [Yosinski et al. (2014)](https://arxiv.org/abs/1411.1792) midieron que las características de las primeras capas son **generales** (sirven casi sin cambios en otro conjunto de datos), mientras que las de las últimas capas son **específicas** de la tarea original.

El transfer learning aprovecha esa generalidad: se toma una red entrenada en una **tarea fuente** con muchos datos y se usan sus pesos como punto de partida para la **tarea destino**. En visión, la tarea fuente habitual es la clasificación de [ImageNet](https://www.image-net.org/) (ILSVRC-2012): 1.28 millones de imágenes de 1000 clases, típicamente a 224×224 píxeles.

Conviene separar la red en dos partes:

- **Backbone** (extractor de características): todas las capas hasta el pooling global. Transforma la imagen en un vector de características; es la parte que se transfiere.
- **Cabeza** (*head*): la capa final que transforma ese vector en logits. Depende de las clases de la tarea, así que **siempre se reemplaza**: ImageNet tiene 1000 clases y CIFAR-10 tiene 10.

Con esa división hay tres estrategias, que vamos a comparar con la misma arquitectura:

![Tres estrategias de entrenamiento](https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/transfer_learning.png)

- **Desde cero**: la referencia; lo mismo que hicimos en la clase 05.
- **Fine-tuning parcial**: se congelan las primeras etapas del backbone, que contienen las características más generales, y se entrenan solo la última etapa y la cabeza. En su versión extrema, llamada *feature extraction*, se congela todo el backbone y solo se entrena la cabeza.
- **Fine-tuning**: se entrena toda la red, pero partiendo de los pesos de ImageNet y con un learning rate pequeño, para ajustarlos sin destruirlos.

## El modelo preentrenado: ResNet-18

[ResNet](https://arxiv.org/abs/1512.03385) es la arquitectura de la **conexión residual** $y = x + F(x)$ que presentamos en la clase 05. **ResNet-18** es la versión más pequeña de la familia: 18 capas con pesos y 11.7 millones de parámetros.

![Arquitectura de ResNet-18](https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/resnet18.png)

El pooling global (`AdaptiveAvgPool2d`, clases 04 y 05) permite que la red acepte imágenes de **cualquier** tamaño: el vector que llega a `fc` siempre tiene 512 valores.

### Cargar un modelo preentrenado

`torchvision` provee [muchas arquitecturas preentrenadas](https://pytorch.org/vision/stable/models.html#classification). Cada una tiene una enumeración de pesos; para ResNet-18 es `ResNet18_Weights`. El objeto `ResNet18_Weights.IMAGENET1K_V1` **no contiene los pesos**: describe un entrenamiento concreto y reúne tres cosas:

| | Qué es | Para qué lo usamos |
|---|---|---|
| URL del archivo de pesos | el resultado del entrenamiento en ImageNet | `resnet18(weights=weights)` lo descarga y lo carga en el modelo |
| `weights.meta` | metadatos: las 1000 clases, la accuracy en ImageNet, la cantidad de parámetros | traducir el índice de la salida a un nombre de clase |
| `weights.transforms()` | el preprocesamiento que se usó para entrenar esos pesos | preparar nuestras imágenes igual que las de ImageNet |

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1  # descripción de los pesos (todavía no se descarga nada)
imagenet_model = resnet18(weights=weights)  # descarga los pesos (~45 MB, la primera vez) y los carga

print(f"Parámetros: {weights.meta['num_params']:,}")
print(f"Accuracy top-1 en ImageNet: {weights.meta['_metrics']['ImageNet-1K']['acc@1']}%")
print(f"Clases: {len(weights.meta['categories'])}, por ejemplo {weights.meta['categories'][:5]}")
print(f"Cabeza: {imagenet_model.fc}")

### ¿Qué ve un modelo de ImageNet en CIFAR-10?

Antes de modificar nada, usamos el modelo tal como viene para clasificar imágenes de CIFAR-10 entre las **1000 clases de ImageNet**. La red solo funciona bien con entradas parecidas a las que vio en el entrenamiento, así que las imágenes tienen que pasar por el mismo preprocesamiento: `weights.transforms()` devuelve un objeto que se aplica como cualquier transformación de `torchvision`.

In [ ]:
DATA_DIR = Path("data")

# CIFAR-10 sin transformaciones (imágenes PIL de 32x32), para visualizar
raw_dataset = datasets.CIFAR10(DATA_DIR, train=True, download=True)
name_classes = raw_dataset.classes
nclasses = len(name_classes)
print(f"Clases de CIFAR-10: {name_classes}")

# Preprocesamiento de ImageNet: redimensiona a 256, recorta el centro de 224x224,
# convierte a tensor float en [0, 1] y normaliza con la media y el desvío de ImageNet
preprocess = weights.transforms()
print(preprocess)

img, label = raw_dataset[0]
x = preprocess(img)
print(f"imagen PIL {img.size} -> tensor {list(x.shape)}")

In [ ]:
# 1. Una imagen de cada clase de CIFAR-10 (la primera de cada clase en el dataset)
sample_idx = [raw_dataset.targets.index(c) for c in range(nclasses)]
images = [raw_dataset[i][0] for i in sample_idx]  # imágenes PIL de 32x32

# 2. Preprocesamiento de ImageNet y un único batch con las 10 imágenes: [10, 3, 224, 224]
batch = torch.stack([preprocess(img) for img in images]).to(DEVICE)

# 3. Forward con el modelo original, sin modificar: un logit por cada clase de ImageNet
imagenet_model = imagenet_model.to(DEVICE).eval()
with torch.no_grad():
    logits = imagenet_model(batch)
print(f"salida del modelo: {list(logits.shape)}")  # [10, 1000]

# 4. Logits -> probabilidades, y las 3 clases de ImageNet más probables para cada imagen
probs = logits.softmax(dim=1)  # cada fila suma 1
top_p, top_c = probs.topk(3, dim=1)  # probabilidades e índices, ambos [10, 3]

# 5. Cada imagen con su clase de CIFAR-10 y las 3 predicciones; meta["categories"] traduce índice -> nombre
fig, axs = plt.subplots(2, 5, figsize=(18, 9))
for ax, img, i, p, c in zip(axs.flat, images, sample_idx, top_p.cpu(), top_c.cpu()):
    ax.imshow(img)
    preds = "\n".join(f"{weights.meta['categories'][ci]} ({pi:.0%})" for pi, ci in zip(p, c))
    ax.set_title(f"CIFAR-10: {name_classes[raw_dataset.targets[i]]}\n{preds}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

El modelo nunca vio CIFAR-10 y recibe imágenes de 32×32 ampliadas a 224×224, con muy poco detalle. Aun así, varias predicciones son correctas con las clases de ImageNet (el caballo como *sorrel*, el camión, el barco como *speedboat*, el auto) y otras son de la categoría correcta aunque no de la clase exacta (el ciervo como *hartebeest*, un antílope; la rana como un lagarto). Los errores (el avión, el pájaro, el gato) se explican en buena medida por la baja resolución. Las características que aprendió el backbone ya contienen gran parte de la información que necesitamos para distinguir las clases de CIFAR-10.

> **¿Por qué no sirve la cabeza?** La capa `fc` tiene **una salida por cada clase de ImageNet**: 1000 logits. No existe una salida para *cat* o *airplane* de CIFAR-10: cada clase de CIFAR-10 está repartida entre muchas clases de ImageNet (más de 100 razas de perros, varios tipos de camiones) y la mayoría de las 1000 no tiene equivalente. Los pesos de `fc` solo sirven para esa tarea. El backbone, en cambio, produce un vector de 512 características que describe la imagen sin depender de las clases.
>
> Por eso **reemplazamos `fc` por una capa nueva `nn.Linear(512, 10)`**, con pesos aleatorios, y la entrenamos con nuestros datos. Es el primer paso de cualquier transfer learning.

### ¿Qué se transfiere? Los filtros de la primera capa

`conv1` tiene 64 filtros de tamaño 3×7×7: cada uno es una pequeña imagen RGB que se puede visualizar directamente. Comparamos los filtros de la red preentrenada con los de una ResNet-18 recién inicializada.

In [ ]:
# conv1.weight: [64, 3, 7, 7] = 64 imágenes RGB de 7x7. make_grid las ubica en una grilla de 8x8;
# normalize + scale_each llevan cada filtro a [0, 1] para poder mostrarlo como imagen
pretrained_filters = make_grid(imagenet_model.conv1.weight.detach().cpu(), nrow=8, padding=1, normalize=True, scale_each=True)
random_filters = make_grid(resnet18(weights=None).conv1.weight.detach().cpu(), nrow=8, padding=1, normalize=True, scale_each=True)

show_tensor_images(
    [pretrained_filters, random_filters],
    titles=["conv1 preentrenada en ImageNet", "conv1 con inicialización aleatoria"],
    figsize=(14, 7),
)

Los filtros preentrenados muestran **bordes orientados** en distintos ángulos, patrones de rayas de distinta frecuencia (parecidos a los [filtros de Gabor](https://en.wikipedia.org/wiki/Gabor_filter) que se usan en procesamiento de imágenes) y **manchas de colores opuestos**. Algunos filtros preentrenados se ven negros: sus pesos son prácticamente 0, el entrenamiento no los utilizó. Los filtros aleatorios no tienen estructura. Son los mismos detectores que implementamos a mano en la clase 04 (el filtro de Sobel), pero aprendidos a partir de los datos. Como cualquier fotografía está formada por bordes, colores y texturas, estos filtros son útiles también para CIFAR-10: es lo que se transfiere.

### Mapas de características a distintas profundidades

Pasamos una imagen por el backbone y guardamos la salida de cada etapa. Mostramos los primeros 6 canales de cada una.

In [ ]:
img = images[7]  # un caballo
x = preprocess(img).unsqueeze(0).to(DEVICE)

stages = {}
with torch.no_grad():
    m = imagenet_model
    x = m.maxpool(m.relu(m.bn1(m.conv1(x))))
    stages["conv1 + maxpool"] = x
    for name in ["layer1", "layer2", "layer3", "layer4"]:
        x = getattr(m, name)(x)
        stages[name] = x

fig, axs = plt.subplots(len(stages), 7, figsize=(14, 10))
for row, (name, fmap) in zip(axs, stages.items()):
    row[0].imshow(img)
    row[0].set_title(f"{name}\n{list(fmap.shape[1:])}", fontsize=10)
    for ax, ch in zip(row[1:], fmap[0, :6].cpu()):
        ax.imshow(ch, cmap="viridis")
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.show()

A medida que la imagen avanza por la red, el alto y el ancho se reducen y la cantidad de canales aumenta (64 → 512). Los mapas de las primeras etapas todavía se parecen a la imagen: resaltan bordes y regiones de un color. Los de `layer4` son de 7×7 y ya no se interpretan a simple vista: cada valor resume una región grande de la imagen y responde a combinaciones de patrones más abstractas. Para una visualización de qué detecta cada capa se recomienda [*Feature Visualization*](https://distill.pub/2017/feature-visualization/) (Olah et al., Distill, 2017).

## Preprocesamiento: normalización y resolución

**Normalización.** En la clase 05 normalizamos con la media y el desvío de CIFAR-10. Con un modelo preentrenado se usan las estadísticas de **ImageNet** (las de `weights.transforms()`): los pesos de `conv1` y las estadísticas de las BatchNorm se ajustaron a entradas con esa escala, y una escala distinta desplazaría las activaciones de todas las capas respecto de lo que la red espera.

**Resolución.** ResNet-18 reduce el alto y el ancho por un factor total de 32 (`conv1`, `maxpool` y el primer bloque de `layer2`, `layer3` y `layer4` tienen `stride=2`). Una imagen de CIFAR-10 de 32×32 llega a `layer4` como un único píxel, y después de `conv1` + `maxpool` ya es de 8×8: los filtros de 7×7 de `conv1` cubren casi un cuarto de la imagen. Los objetos de CIFAR-10 quedan mucho más pequeños que los objetos para los que la red aprendió sus filtros.

Ampliar la imagen no agrega información (es una interpolación), pero lleva los objetos a una escala más parecida a la de ImageNet. El costo crece con la cantidad de píxeles: a 128×128 cada imagen requiere 16 veces más cómputo que a 32×32. Usamos **128×128** como compromiso entre accuracy y tiempo; más adelante lo justificamos con un experimento.

In [ ]:
backbone_only = nn.Sequential(*list(imagenet_model.children())[:-2])  # hasta layer4, sin avgpool ni fc

resized = []
for size in [32, 64, 128, 224]:
    with torch.no_grad():
        out = backbone_only(torch.zeros(1, 3, size, size, device=DEVICE))
    print(f"entrada {size:3d}x{size:<3d} -> salida de layer4: {list(out.shape[1:])}")
    resized.append(T.Resize(size, antialias=True)(T.ToImage()(img)))

show_tensor_images(resized, titles=[f"{r.shape[-1]}x{r.shape[-1]}" for r in resized], figsize=(14, 4))

## Dataset y DataLoader

Usamos la misma función `get_dataloaders` de la clase 05: el conjunto de entrenamiento se carga **dos veces** (una con data augmentation para train y otra sin augmentation para validación) y se divide con una única permutación de índices y [`Subset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset). Agregamos un parámetro `n_train` para entrenar, más adelante, con solo una parte de las imágenes; validación y test no cambian.

Las transformaciones son las de la clase 05 (traslación de hasta 4 píxeles y reflejo horizontal, aplicadas sobre la imagen de 32×32), seguidas del cambio de tamaño a 128×128 y la normalización de ImageNet.

In [ ]:
IMG_SIZE = 128

# media y desvío por canal (R, G, B) de ImageNet: los mismos de weights.transforms()
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def make_transforms(img_size, augment):
    # El data augmentation va primero: actúa sobre la imagen original de 32x32 con los mismos parámetros
    # que en la clase 05 (padding=4 es una traslación de hasta 4 de 32 píxeles; después del Resize a 128
    # haría falta padding=16 para el mismo efecto). Normalize va siempre al final, sobre el tensor float.
    augmentation = [T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(p=0.5)] if augment else []
    return T.Compose(
        augmentation
        + [
            T.ToImage(),  # PIL -> tensor uint8 [C, H, W]
            T.Resize(img_size, antialias=True),  # 32x32 -> img_size x img_size
            T.ToDtype(torch.float32, scale=True),  # uint8 [0, 255] -> float32 [0, 1]
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


train_transform = make_transforms(IMG_SIZE, augment=True)
# Validación y test: nunca llevan data augmentation
test_transform = make_transforms(IMG_SIZE, augment=False)

In [ ]:
def get_dataloaders(
    train_transf, test_transf, n_train=None, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
):
    """
    Función para obtener los dataloaders de entrenamiento, validación y test

    Args:
    - train_transf: transformaciones para el dataset de entrenamiento
    - test_transf: transformaciones para los datasets de validación y test
    - n_train: si no es None, se usan solo las primeras n_train imágenes del split de entrenamiento
    - batch_size: tamaño del batch
    - num_workers: número de workers para cargar los datos
    """

    # El conjunto de entrenamiento se carga dos veces: las mismas imágenes con distinto transform
    train_base = datasets.CIFAR10(
        DATA_DIR, train=True, download=True, transform=train_transf
    )
    val_base = datasets.CIFAR10(
        DATA_DIR, train=True, download=True, transform=test_transf
    )
    test_dataset = datasets.CIFAR10(
        DATA_DIR, train=False, download=True, transform=test_transf
    )

    # dividimos los índices en entrenamiento (80%) y validación (20%)
    indices = torch.randperm(
        len(train_base),
        generator=torch.Generator().manual_seed(SEED),  # mismo split en cada llamada
    )
    train_size = int(0.8 * len(train_base))
    train_idx = indices[:train_size]
    if n_train is not None:
        train_idx = train_idx[:n_train]  # validación y test no cambian
    train_dataset = Subset(train_base, train_idx.tolist())
    val_dataset = Subset(val_base, indices[train_size:].tolist())

    # creamos los dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_dataloaders(train_transform, test_transform)

## Tres estrategias con la misma arquitectura

Para que la comparación sea justa, las tres estrategias usan la misma ResNet-18, los mismos datos, la misma cantidad máxima de épocas y early stopping sobre la pérdida de validación. Lo único que cambia es el punto de partida de los pesos y qué parte de la red se entrena.

La función `run_experiment` entrena un modelo con `train` de `utils.py`, mide el tiempo y guarda la accuracy en validación. El optimizador recibe **solo los parámetros entrenables** (`requires_grad=True`).

In [ ]:
# definicion de hiperparametros
EPOCHS = 10
PATIENCE = 3
criterion = nn.CrossEntropyLoss()

results = {}  # nombre -> métricas en validación y curvas de entrenamiento


def run_experiment(name, model, lr, train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE, verbose=True):
    model = model.to(DEVICE)
    trainable = [p for p in model.parameters() if p.requires_grad]  # los congelados no van al optimizador
    optimizer = optim.Adam(trainable, lr=lr)

    start = time.perf_counter()
    train_errors, val_errors = train(
        model=model,
        optimizer=optimizer,
        criterion=criterion,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        do_early_stopping=True,
        patience=patience,
        epochs=epochs,
        log_fn=print_log if verbose else None,
    )
    elapsed = time.perf_counter() - start

    results[name] = {
        "val_acc": accuracy(model, val_loader, DEVICE),  # con los pesos de la mejor época (restore_best)
        "trainable": sum(p.numel() for p in trainable),
        "sec_per_epoch": elapsed / len(train_errors),
        "train_errors": train_errors,
        "val_errors": val_errors,
    }
    print(f"{name}: accuracy en validación {results[name]['val_acc']:.2%}, {results[name]['sec_per_epoch']:.1f} s por época")
    return model

### Estrategia 1: desde cero

`resnet18(weights=None)` construye la misma arquitectura con pesos aleatorios. El argumento `num_classes=10` crea directamente la cabeza de 10 salidas. Usamos el learning rate de la clase 05 para Adam, `1e-3`.

In [ ]:
scratch_model = resnet18(weights=None, num_classes=nclasses)
scratch_model = run_experiment("desde cero", scratch_model, lr=1e-3, train_loader=train_loader, val_loader=val_loader)

### Estrategia 2: fine-tuning parcial

**Congelar** un parámetro es poner `requires_grad=False`: autograd no calcula su gradiente y el optimizador no lo modifica. Congelamos toda la red excepto las dos últimas capas: `layer4`, la última etapa del backbone, y la cabeza nueva.

- `conv1` a `layer3` detectan características generales (bordes, texturas, partes) que sirven tal como están.
- `layer4` produce las características más específicas de ImageNet: es la parte del backbone que más conviene adaptar.
- La cabeza nueva se crea **después** de congelar, así que sus parámetros tienen `requires_grad=True`.

<small>Nota: `requires_grad=False` no detiene la actualización de las estadísticas de BatchNorm (`running_mean`, `running_var`) en `model.train()`. Para un congelamiento estricto, las capas congeladas se ponen en modo `eval`; aquí lo omitimos para simplificar.</small>

In [ ]:
def partial_finetuning_model():
    model = resnet18(weights=weights)
    for param in model.parameters():
        param.requires_grad = False  # congelamos toda la red
    for param in model.layer4.parameters():
        param.requires_grad = True  # descongelamos la última etapa del backbone
    model.fc = nn.Linear(model.fc.in_features, nclasses)  # cabeza nueva: se crea con requires_grad=True
    return model


summary(partial_finetuning_model(), input_size=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE), depth=1)

En el `summary`, las capas congeladas aparecen con sus parámetros entre paréntesis. `layer4` concentra 8.4 de los 11.2 millones de parámetros (tiene 512 canales), pero el entrenamiento igual es más rápido que el de la red completa: no hace falta calcular gradientes para `conv1` a `layer3`.

Antes del entrenamiento completo, un experimento corto que justifica la resolución elegida: entrenamos 2 épocas con imágenes de 32×32, 64×64 y 128×128.

In [ ]:
for size in [32, 64, 128]:
    tr, va, _ = get_dataloaders(make_transforms(size, augment=True), make_transforms(size, augment=False))
    run_experiment(f"fine-tuning parcial {size}x{size}", partial_finetuning_model(), lr=1e-4,
                   train_loader=tr, val_loader=va, epochs=2, verbose=False)

A 32×32 las características de ImageNet son poco útiles: los objetos son mucho más pequeños que los que la red aprendió a reconocer. Con solo 2 épocas, la accuracy en validación pasa de ≈ 58% a 32×32 a ≈ 79% a 64×64 y ≈ 90% a 128×128. A 224×224 (ejercicio 4) cada época es casi 3 veces más lenta que a 128×128.

Ahora entrenamos con el mismo presupuesto que el modelo desde cero. El learning rate es **10 veces menor** (`1e-4`); la razón se explica en la estrategia 3.

In [ ]:
pft_model = run_experiment("fine-tuning parcial", partial_finetuning_model(), lr=1e-4,
                           train_loader=train_loader, val_loader=val_loader)

### Estrategia 3: fine-tuning

Ahora se entrena toda la red, partiendo de los pesos de ImageNet. Solo se reemplaza la cabeza.

El learning rate es `1e-4`, 10 veces menor que desde cero. Los pesos preentrenados ya son una buena solución, y en las primeras iteraciones los gradientes provienen de una cabeza aleatoria: con un learning rate grande esos gradientes modifican bruscamente el backbone y se pierde parte de lo aprendido en ImageNet (*catastrophic forgetting*). El ejercicio 2 lo compara con `1e-3`.

In [ ]:
ft_model = resnet18(weights=weights)
ft_model.fc = nn.Linear(ft_model.fc.in_features, nclasses)  # cabeza nueva, pesos aleatorios

ft_model = run_experiment("fine-tuning", ft_model, lr=1e-4, train_loader=train_loader, val_loader=val_loader)

## Comparación en validación

Comparamos los modelos en **validación**; test lo reservamos para el modelo final (mismo criterio que en las clases 03 y 05).

In [ ]:
def show_results(names):
    print(f"{'estrategia':<32}{'val acc':>9}{'entrenables':>14}{'s/época':>10}{'épocas':>8}")
    for name in names:
        r = results[name]
        print(f"{name:<32}{r['val_acc']:>9.2%}{r['trainable']:>14,}{r['sec_per_epoch']:>10.1f}{len(r['val_errors']):>8}")


def plot_curves(names, title):
    plt.figure(figsize=(10, 5))
    colors = {"desde cero": "tab:red", "fine-tuning parcial": "tab:green", "fine-tuning": "tab:blue"}
    for name in names:
        color = colors[name.split(",")[0]]  # mismo color por estrategia en todos los gráficos
        plt.plot(results[name]["train_errors"], "--", color=color, label=f"train {name}")
        plt.plot(results[name]["val_errors"], "-", color=color, label=f"val {name}")
    plt.title(title)
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


strategies = ["desde cero", "fine-tuning parcial", "fine-tuning"]
show_results(strategies)
plot_curves(strategies, "ResNet-18 en CIFAR-10: tres estrategias")

**¿Qué muestran los resultados?**

- **Fine-tuning** es la mejor estrategia. Desde la **primera** época su pérdida de validación es menor que la mejor que alcanza el modelo desde cero. Supera también a la DenseNet de la clase 05 (≈ 84%) con un tercio de sus épocas.
- **Fine-tuning parcial** queda unos 3 puntos por debajo, pero cada época tarda casi la mitad: no se calculan gradientes para `conv1` a `layer3`. Las características generales de esas etapas sirven sin cambios; alcanza con adaptar `layer4` y la cabeza.
- **Desde cero** la misma arquitectura necesita todas las épocas para llegar a un resultado inferior: tiene que aprender desde los bordes de la primera capa hasta las clases, solo con 40,000 imágenes.

## ¿Cuándo conviene? Entrenamiento con pocos datos

CIFAR-10 tiene 5,000 imágenes de entrenamiento por clase. En muchos problemas reales la cantidad de imágenes es mucho menor. Repetimos la comparación entre desde cero y fine-tuning con solo **2,000 imágenes** de entrenamiento (≈ 200 por clase); el conjunto de validación es el mismo. Con tan pocos datos cada época es corta, así que permitimos más épocas.

In [ ]:
N_SMALL = 2000
small_train_loader, _, _ = get_dataloaders(train_transform, test_transform, n_train=N_SMALL)

run_experiment(f"desde cero, {N_SMALL} imágenes", resnet18(weights=None, num_classes=nclasses), lr=1e-3,
               train_loader=small_train_loader, val_loader=val_loader, epochs=20, patience=5, verbose=False)

small_ft = resnet18(weights=weights)
small_ft.fc = nn.Linear(small_ft.fc.in_features, nclasses)
run_experiment(f"fine-tuning, {N_SMALL} imágenes", small_ft, lr=1e-4,
               train_loader=small_train_loader, val_loader=val_loader, epochs=20, patience=5, verbose=False)

small = [f"desde cero, {N_SMALL} imágenes", f"fine-tuning, {N_SMALL} imágenes"]
show_results(strategies + small)
plot_curves(small, f"ResNet-18 con {N_SMALL} imágenes de entrenamiento")

Con 20 veces menos datos la diferencia se amplía mucho. El modelo desde cero muestra una brecha grande entre train y validación y una pérdida de validación inestable: 11 millones de parámetros aprendidos desde cero con 2,000 imágenes. El fine-tuning con 2,000 imágenes también sobreajusta (la pérdida de train llega casi a 0), pero su accuracy en validación **supera** a la del modelo desde cero entrenado con las 40,000. Las características de ImageNet reemplazan buena parte de los datos que faltan. Es el escenario en el que el transfer learning resulta más valioso.

## Evaluación final en test

Elegimos el modelo de fine-tuning con todos los datos, el mejor en validación, y lo evaluamos **una sola vez** en test.

In [ ]:
model_classification_report(
    ft_model, test_loader, DEVICE, nclasses, do_confusion_matrix=True, target_names=name_classes
)

El resultado en test es consistente con el de validación. Como en la clase 05, en la matriz de confusión los errores se concentran en **cat ↔ dog**, seguidos de **automobile → truck** y **ship → airplane**.

## Conclusiones

- Un backbone entrenado en ImageNet contiene características generales (bordes, colores, texturas, partes) que sirven para otros problemas de visión. Transferirlas mejora la accuracy, acelera el entrenamiento y reduce la cantidad de datos necesaria.
- Al usar un modelo preentrenado hay que respetar sus condiciones de entrenamiento: la misma **normalización** y una **resolución** comparable. La cabeza siempre se reemplaza.
- Congelar capas (`requires_grad=False`) reduce los parámetros que se entrenan y el costo de cada época. Las que conviene congelar son las primeras, las más generales.
- El fine-tuning requiere un learning rate menor que el entrenamiento desde cero, para ajustar los pesos preentrenados sin destruirlos.

Una guía habitual para elegir la estrategia (ver las notas de [CS231n](https://cs231n.github.io/transfer-learning/)) depende de la cantidad de datos y del parecido con ImageNet:

| | Dominio parecido a ImageNet (fotografías) | Dominio distinto (imágenes médicas, satelitales, espectrogramas) |
|---|---|---|
| **Pocos datos** | Feature extraction, o fine-tuning con learning rate pequeño y pocas épocas | Lo más difícil: fine-tuning con mucha regularización, o usar características de capas intermedias, más generales |
| **Muchos datos** | Fine-tuning | Fine-tuning; la ventaja sobre entrenar desde cero es menor, pero la convergencia suele ser más rápida |

Con dominios muy distintos el beneficio no está garantizado: [Raghu et al. (2019)](https://arxiv.org/abs/1902.07208) muestran que en imágenes médicas modelos pequeños entrenados desde cero obtienen resultados comparables, y [He et al. (2019)](https://arxiv.org/abs/1811.08883) que con suficientes datos y épocas el entrenamiento desde cero alcanza al fine-tuning.

## Ejercicios

1. **Feature extraction**: congelar también `layer4` y entrenar solo la cabeza (5,130 parámetros). ¿Cuánto cae la accuracy? Repetir con el backbone en modo `eval` durante el entrenamiento (ver la nota de BatchNorm de la estrategia 2): ¿qué cambia?
2. **Learning rate del fine-tuning**: repetir la estrategia 3 con `lr=1e-3`. Comparar las curvas y la accuracy con `1e-4`.
3. **Learning rates por grupo**: un optimizador de PyTorch acepta [grupos de parámetros](https://pytorch.org/docs/stable/optim.html#per-parameter-options) con un learning rate distinto cada uno. Entrenar con `1e-3` para la cabeza y `1e-4` para el backbone: `optim.Adam([{"params": model.fc.parameters(), "lr": 1e-3}, {"params": backbone_params, "lr": 1e-4}])`.
4. **Resolución**: repetir el fine-tuning con `IMG_SIZE = 224`. ¿Cuánto mejora la accuracy y cuánto aumenta el tiempo por época?
5. **Descongelar más etapas**: repetir la estrategia 2 descongelando también `layer3`. ¿Dónde queda respecto del fine-tuning parcial y del completo?
6. **Otra arquitectura**: repetir las estrategias 2 y 3 con `densenet121` (la cabeza es `classifier`) o con una red pensada para dispositivos móviles como `mobilenet_v3_small` o `efficientnet_b0`. Comparar accuracy, parámetros y tiempo por época.
7. **Curva de datos**: repetir el experimento con pocos datos para `N_SMALL` = 500, 2,000 y 10,000, y graficar la accuracy en validación de ambas estrategias en función de la cantidad de imágenes.

#### Lectura adicional

- Yosinski et al., [*How transferable are features in deep neural networks?*](https://arxiv.org/abs/1411.1792) (2014).
- Zeiler y Fergus, [*Visualizing and Understanding Convolutional Networks*](https://arxiv.org/abs/1311.2901) (2014).
- Kornblith et al., [*Do Better ImageNet Models Transfer Better?*](https://arxiv.org/abs/1805.08974) (2019).
- CS231n, [*Transfer Learning*](https://cs231n.github.io/transfer-learning/).
- Tutorial de PyTorch: [*Transfer Learning for Computer Vision*](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html).
- Documentación de [modelos preentrenados de `torchvision`](https://pytorch.org/vision/stable/models.html).